<a href="https://colab.research.google.com/github/syedmahmoodiagents/NLP/blob/main/SimpleRNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
# import torch.nn.functional as F
import torch.optim as optim

In [ ]:
from torch.nn.utils.rnn import pad_sequence

In [ ]:
from gensim.models import Word2Vec
# from gensim.test.utils import common_texts

In [ ]:
# common_texts

In [ ]:
corpus = [
    "I love machine learning",
    "wordvec is great algorithm",
    "Implementing word2vec is fun"
]

In [ ]:
seq = [sen.split() for sen in corpus]

In [ ]:
seq

[['I', 'love', 'machine', 'learning'],
 ['wordvec', 'is', 'great', 'algorithm'],
 ['Implementing', 'word2vec', 'is', 'fun']]

In [ ]:
mod = Word2Vec(seq, vector_size=20, window=2, min_count=1)

In [ ]:
keyind = mod.wv.key_to_index

In [ ]:
keyind

{'is': 0,
 'fun': 1,
 'word2vec': 2,
 'Implementing': 3,
 'algorithm': 4,
 'great': 5,
 'wordvec': 6,
 'learning': 7,
 'machine': 8,
 'love': 9,
 'I': 10}

In [ ]:
keyind['love']

9

In [ ]:
mod.wv['love']

array([-0.0468503 ,  0.0191337 ,  0.0244224 , -0.03214282,  0.00604279,
       -0.01037438,  0.00012202, -0.04941754,  0.01346002, -0.02375053,
        0.00543823, -0.00788112,  0.01098337, -0.03940788, -0.01358592,
        0.01331599,  0.02673341, -0.01195757, -0.04755047,  0.02252939],
      dtype=float32)

In [ ]:
emb = mod.wv.vectors

In [ ]:
emb.shape

(11, 20)

In [ ]:
# mod.wv['love']

In [ ]:
input_data = [[keyind[word] for word in sentence] for sentence in seq]

In [ ]:
input_data

[[10, 9, 8, 7], [6, 0, 5, 4], [3, 2, 0, 1]]

In [ ]:
# nn.Embedding.from_pretrained(torch.FloatTensor(emb))

In [ ]:
class MyModel(nn.Module):
    def __init__(self, embed_matrix):
        super(MyModel, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(embed_matrix)
        self.rnn = nn.RNN(20, 50, batch_first=True, num_layers=7)
        self.fc = nn.Linear(50, 11)

    def forward(self, x):
        print("before embedding -> ", x.shape)
        x = self.embedding(x)
        print("after embedding -> ", x.shape)
        x, h = self.rnn(x)
        print("after rnn x and h -> ", x.shape, h.shape)
        print("hidden -> ", x[:, -1, :].shape, h[-1].shape)
        x = self.fc(x[:, -1, :])
        print("after fc ->", x.shape)
        return x

In [ ]:
ml = MyModel(torch.FloatTensor(emb))

In [ ]:
ml(torch.LongTensor(input_data))

before embedding ->  torch.Size([3, 4])
after embedding ->  torch.Size([3, 4, 20])
after rnn x and h ->  torch.Size([3, 4, 50]) torch.Size([7, 3, 50])
hidden ->  torch.Size([3, 50]) torch.Size([3, 50])
after fc -> torch.Size([3, 11])


torch.Size([3, 11])

In [ ]:
class NewModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(11, 20)
        # self.rnn = nn.RNN(20, 50, batch_first=True)
        self.rnn = nn.LSTM(20, 50, batch_first=True)
        self.fc = nn.Linear(50, 11)

    def forward(self, x):
        x = self.embedding(x)
        # x, h = self.rnn(x)
        x, (h, c) = self.rnn(x)
        x = self.fc(x[:, -1, :])
        return x

In [ ]:
mld = NewModel()

In [ ]:
yp = mld(torch.LongTensor(input_data))

In [ ]:
import torch.nn.functional as F

In [ ]:
yp.argmax(dim=1)

tensor([4, 4, 5])

In [ ]:
softy = F.softmax(yp, dim=1)

In [ ]:
import numpy as np

In [ ]:
ar = softy.detach().numpy()

In [ ]:
np.argmax(ar, axis=1)

array([4, 4, 5])

In [ ]:
mld.eval()
with torch.no_grad():
    input_sequence = torch.LongTensor([6, 0, 5]).unsqueeze(0)  # Example input sequence (batch size = 1)
    output = mld(input_sequence)

In [ ]:
output.argmax(dim=1)

tensor([9])

# Spacy

In [ ]:
import spacy


In [ ]:



# Load the spaCy model
nlp = spacy.load("en_core_web_sm")

# Gather all English words
vocab = set(nlp.vocab.strings)
vocab = sorted(vocab)  # Sort for consistent indexing

# Create word to index and index to word mappings
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for idx, word in enumerate(vocab)}

# Example sentences
sentences = [
    ['i', 'love', 'machine', 'learning'],
    ['deep', 'learning', 'is', 'fun'],
    ['natural', 'language', 'processing'],
    ['neural', 'networks', 'are', 'powerful']
]

# Convert sentences to indices
sentences_idx = [[word_to_idx[word] for word in sentence] for sentence in sentences]

# Pad the sequences
padded_sequences = pad_sequence([torch.tensor(seq) for seq in sentences_idx], batch_first=True, padding_value=0)

# Define the RNN model
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(RNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        x, hidden = self.rnn(x)
        x = self.fc(x[:, -1, :])
        return x, hidden

    def predict_next_word(self, x):
        self.eval()
        with torch.no_grad():
            x = self.embedding(x)
            x, hidden = self.rnn(x)
            output = self.fc(hidden[-1])
            predicted_word_idx = torch.argmax(output, dim=1)
        return predicted_word_idx.item()

# Define parameters
vocab_size = len(vocab)
embedding_dim = 100
hidden_dim = 128
output_dim = vocab_size

# Instantiate the model
model = RNNModel(vocab_size, embedding_dim, hidden_dim, output_dim)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

# Training loop (just one example epoch)
model.train()
optimizer.zero_grad()
inputs = padded_sequences[:, :-1]
targets = padded_sequences[:, 1:]
outputs, _ = model(inputs)
loss = criterion(outputs.view(-1, output_dim), targets.view(-1))
loss.backward()
optimizer.step()
print(f"Loss: {loss.item()}")

# Predict the next word using the trained model
model.eval()
with torch.no_grad():
    input_sequence = torch.tensor([word_to_idx['i'], word_to_idx['love'], word_to_idx['machine']]).unsqueeze(0)
    next_word_idx = model.predict_next_word(input_sequence)
    print(f"Predicted Next Word: {idx_to_word[next_word_idx]}")
